# 05 — Ablation Analysis

**Input:** `../data/processed/pm_day_features.csv`, `../data/processed/embeddings.npy`
**Output:** `../outputs/tables/nested_ablation_*.csv`, `../outputs/tables/state_trait_glm.csv`

**Description:**
- Nested ablation: content vs engagement vs instability vs constriction
- Four text feature categories:
  - **Content (C):** PCA on sentence-transformer embeddings — *what* people write
  - **Engagement (E):** word count features — *how much* people write
  - **Instability (I):** cosine distance to person centroid — *how different* from their average
  - **Constriction (K):** type-token ratio, root TTR — *how restricted* their vocabulary
- Incremental utility: does text add value beyond numeric crisis ratings?
- CV-safe instability (centroids from training folds only)
- State vs trait decomposition via cluster-robust GLM

In [16]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
import statsmodels.api as sm

# =========================
# CONFIG
# =========================
DATA_PATH = os.path.join("..", "data", "processed", "pm_day_features.csv")
EMBED_PATH = os.path.join("..", "data", "processed", "embeddings.npy")
OUT_DIR = os.path.join("..", "outputs", "tables")
os.makedirs(OUT_DIR, exist_ok=True)

PID_COL = "expiwell_id_clean"
TEXT_COL = "pm_day_text"
CRISIS_COL = "crisis_PM_from_full"

N_PCS = 20
RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)

In [18]:
# =========================
# LOAD
# =========================
pm_day = pd.read_csv(DATA_PATH)
X_text = np.load(EMBED_PATH)

assert X_text.shape[0] == len(pm_day)
print("Loaded data:", pm_day.shape)

# Prepare feature blocks
X_eng = pm_day[["log1p_wc", "wc_le1"]].values.astype(float)
X_crisis = pm_day[[CRISIS_COL]].astype(float).values

# Constriction features — impute NaN (minimal text) with column median
# NaN occurs for responses with <=1 word where TTR is undefined
X_constrict_df = pm_day[["ttr", "root_ttr"]].copy()
X_constrict_df = X_constrict_df.fillna(X_constrict_df.median())
X_constrict = X_constrict_df.values.astype(float)

groups = pm_day[PID_COL].astype(str).values

mask_ok = (
    np.isfinite(X_crisis).all(axis=1) &
    np.isfinite(X_eng).all(axis=1) &
    np.isfinite(X_constrict).all(axis=1)
)
print(f"Rows retained: {mask_ok.sum()} / {len(pm_day)}")
print(f"Constriction features: ttr, root_ttr (NaN imputed with median)")

Loaded data: (2511, 73)
Rows retained: 2511 / 2511
Constriction features: ttr, root_ttr (NaN imputed with median)


In [20]:
# =========================
# HELPERS
# =========================
def l2_normalize_rows(X, eps=1e-12):
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(n, eps)


def fit_predict_logit(Xtr, Xte, ytr, C=1.0, random_state=0):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("logit", LogisticRegression(
            penalty="l2", C=C, solver="liblinear",
            max_iter=5000, random_state=random_state
        ))
    ])
    pipe.fit(Xtr, ytr)
    return pipe.predict_proba(Xte)[:, 1]

In [22]:
# =========================
# NESTED ABLATION (with CV-safe instability, 4 feature categories)
# =========================
def run_nested_ablation(X_text, X_eng, X_constrict, X_crisis, y, groups, n_pcs=20):
    gkf = GroupKFold(n_splits=5)
    X_text_norm = l2_normalize_rows(X_text)

    # Model naming convention:
    #   C = Content (PCA), E = Engagement (wc), I = Instability (cosdist),
    #   K = Constriction (TTR), N = Numeric baseline
    model_names = [
        # Text-only models (building up)
        "C",          # content only
        "CE",         # + engagement
        "CK",         # + constriction (skipping engagement)
        "CEK",        # content + engagement + constriction
        "CEI",        # content + engagement + instability (no constriction)
        "CEIK",       # all text features
        # Numeric baseline models (building up)
        "N",          # numeric only
        "NC",         # + content
        "NCK",        # + content + constriction
        "NCE",        # + content + engagement
        "NCEK",       # + content + engagement + constriction
        "NCEI",       # + content + engagement + instability
        "NCEIK",      # numeric + all text features
    ]
    oof = {k: np.full(len(y), np.nan) for k in model_names}

    for tr, te in gkf.split(X_text, y, groups):
        # PCA on text (fit on train only)
        pca = PCA(n_components=min(n_pcs, X_text.shape[1]), random_state=RANDOM_SEED)
        Ttr = pca.fit_transform(X_text[tr])
        Tte = pca.transform(X_text[te])

        # CV-safe instability (centroids from train only)
        tr_idx, te_idx = np.array(tr), np.array(te)
        train_groups = groups[tr_idx]
        global_centroid = X_text_norm[tr_idx].mean(axis=0)
        global_centroid /= max(np.linalg.norm(global_centroid), 1e-12)

        pid_to_centroid = {}
        for pid in np.unique(train_groups):
            rows = X_text_norm[tr_idx[train_groups == pid]]
            c = rows.mean(axis=0)
            c /= max(np.linalg.norm(c), 1e-12)
            pid_to_centroid[pid] = c

        instab_tr = np.array([1.0 - X_text_norm[idx] @ pid_to_centroid.get(groups[idx], global_centroid)
                              for idx in tr_idx]).reshape(-1, 1)
        instab_te = np.array([1.0 - X_text_norm[idx] @ pid_to_centroid.get(groups[idx], global_centroid)
                              for idx in te_idx]).reshape(-1, 1)

        # Shorthand for feature matrices
        Etr, Ete = X_eng[tr], X_eng[te]
        Ktr, Kte = X_constrict[tr], X_constrict[te]
        Ntr, Nte = X_crisis[tr], X_crisis[te]
        Itr, Ite = instab_tr, instab_te

        # --- Text-only models ---
        oof["C"][te] = fit_predict_logit(Ttr, Tte, y[tr])
        oof["CE"][te] = fit_predict_logit(
            np.hstack([Ttr, Etr]), np.hstack([Tte, Ete]), y[tr])
        oof["CK"][te] = fit_predict_logit(
            np.hstack([Ttr, Ktr]), np.hstack([Tte, Kte]), y[tr])
        oof["CEK"][te] = fit_predict_logit(
            np.hstack([Ttr, Etr, Ktr]), np.hstack([Tte, Ete, Kte]), y[tr])
        oof["CEI"][te] = fit_predict_logit(
            np.hstack([Ttr, Etr, Itr]), np.hstack([Tte, Ete, Ite]), y[tr])
        oof["CEIK"][te] = fit_predict_logit(
            np.hstack([Ttr, Etr, Itr, Ktr]), np.hstack([Tte, Ete, Ite, Kte]), y[tr])

        # --- Numeric baseline models ---
        oof["N"][te] = fit_predict_logit(Ntr, Nte, y[tr])
        oof["NC"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr]), np.hstack([Nte, Tte]), y[tr])
        oof["NCK"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr, Ktr]), np.hstack([Nte, Tte, Kte]), y[tr])
        oof["NCE"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr, Etr]), np.hstack([Nte, Tte, Ete]), y[tr])
        oof["NCEK"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr, Etr, Ktr]), np.hstack([Nte, Tte, Ete, Kte]), y[tr])
        oof["NCEI"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr, Etr, Itr]), np.hstack([Nte, Tte, Ete, Ite]), y[tr])
        oof["NCEIK"][te] = fit_predict_logit(
            np.hstack([Ntr, Ttr, Etr, Itr, Ktr]), np.hstack([Nte, Tte, Ete, Ite, Kte]), y[tr])

    # Build results with readable labels
    label_map = {
        "C":     "Content",
        "CE":    "Content + Engagement",
        "CK":    "Content + Constriction",
        "CEK":   "Content + Engagement + Constriction",
        "CEI":   "Content + Engagement + Instability",
        "CEIK":  "Content + Engagement + Instability + Constriction",
        "N":     "Numeric only",
        "NC":    "Numeric + Content",
        "NCK":   "Numeric + Content + Constriction",
        "NCE":   "Numeric + Content + Engagement",
        "NCEK":  "Numeric + Content + Engagement + Constriction",
        "NCEI":  "Numeric + Content + Engagement + Instability",
        "NCEIK": "Numeric + All text features",
    }

    rows = []
    for name, p in oof.items():
        rows.append({
            "model": name,
            "label": label_map[name],
            "AUROC": roc_auc_score(y, p),
            "AUPRC": average_precision_score(y, p),
        })
    return pd.DataFrame(rows)

In [24]:
# =========================
# RUN ABLATION FOR EACH OUTCOME
# =========================
for outcome in ["high_any_item_eq3", "moderate_total_ge2", "any_risk_total_gt0"]:
    y = pm_day[outcome].astype(int).values
    y_m = y[mask_ok]
    X_text_m = X_text[mask_ok]
    X_eng_m = X_eng[mask_ok]
    X_constrict_m = X_constrict[mask_ok]
    X_crisis_m = X_crisis[mask_ok]
    groups_m = groups[mask_ok]

    if y_m.sum() < 20:
        print(f"Skipping {outcome}: too few positives")
        continue

    print("")
    print("=" * 60)
    print(f"Outcome: {outcome} | N={len(y_m)} | pos={y_m.sum()} ({y_m.mean():.3f})")
    print("=" * 60)

    results = run_nested_ablation(
        X_text_m, X_eng_m, X_constrict_m, X_crisis_m, y_m, groups_m, n_pcs=N_PCS
    )

    # Display in two blocks for readability
    text_only = results[~results["model"].str.startswith("N")]
    with_numeric = results[results["model"].str.startswith("N")]

    print("")
    print("--- Text-only models ---")
    print(text_only[["model", "label", "AUROC", "AUPRC"]].to_string(index=False))
    print("")
    print("--- With numeric baseline ---")
    print(with_numeric[["model", "label", "AUROC", "AUPRC"]].to_string(index=False))

    out_path = os.path.join(OUT_DIR, f"nested_ablation_{outcome}.csv")
    results.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")


Outcome: high_any_item_eq3 | N=2511 | pos=99 (0.039)

--- Text-only models ---
model                                             label    AUROC    AUPRC
    C                                           Content 0.759025 0.202771
   CE                              Content + Engagement 0.735942 0.188852
   CK                            Content + Constriction 0.754686 0.183254
  CEK               Content + Engagement + Constriction 0.746419 0.154687
  CEI                Content + Engagement + Instability 0.735573 0.150657
 CEIK Content + Engagement + Instability + Constriction 0.746826 0.141334

--- With numeric baseline ---
model                                         label    AUROC    AUPRC
    N                                  Numeric only 0.632312 0.064029
   NC                             Numeric + Content 0.746941 0.246399
  NCK              Numeric + Content + Constriction 0.744746 0.230422
  NCE                Numeric + Content + Engagement 0.718631 0.225609
 NCEK Numeric + Conte

In [25]:
# =========================
# STATE vs TRAIT GLM (cluster-robust)
# =========================
# Now includes constriction (ttr, root_ttr) in the GLM
N_PCS_CONTENT = 5
OUTCOME_BIN = "high_any_item_eq3"
MECH_COLS = (
    ["log1p_wc", "wc_le1", "instability_cosdist", "ttr", "root_ttr"] +
    [f"PC{i}" for i in range(1, N_PCS_CONTENT + 1)]
)

X_cols = []
for c in MECH_COLS:
    X_cols += [f"{c}_within", f"{c}_between"]

dfm = pm_day.copy()
y_glm = dfm[OUTCOME_BIN].astype(float).values
mask_glm = np.isfinite(y_glm)
for c in list(X_cols):
    if c in dfm.columns:
        mask_glm &= np.isfinite(dfm[c].values)
    else:
        print(f"Warning: {c} not in data, skipping from GLM")
        X_cols = [x for x in X_cols if x != c]

dfm = dfm.loc[mask_glm].copy()
y_glm = dfm[OUTCOME_BIN].astype(int).values
X_glm = sm.add_constant(dfm[X_cols].astype(float), has_constant="add")
groups_glm = dfm[PID_COL].astype(str).values

glm = sm.GLM(y_glm, X_glm, family=sm.families.Binomial())
res = glm.fit(cov_type="cluster", cov_kwds={"groups": groups_glm})

print("")
print("=== STATE vs TRAIT (cluster-robust GLM) ===")
print(f"Outcome: {OUTCOME_BIN} | N={len(dfm)} | participants={dfm[PID_COL].nunique()}")
print(res.summary())

# Save table
rows = []
for c in MECH_COLS:
    for part in ["within", "between"]:
        name = f"{c}_{part}"
        if name in res.params.index:
            rows.append({
                "feature": c, "component": part,
                "beta": float(res.params[name]),
                "SE": float(res.bse[name]),
                "z": float(res.tvalues[name]),
                "p": float(res.pvalues[name]),
                "OR": float(np.exp(res.params[name])),
                "CI_lo": float(res.conf_int().loc[name, 0]),
                "CI_hi": float(res.conf_int().loc[name, 1]),
            })

tab = pd.DataFrame(rows)
tab_path = os.path.join(OUT_DIR, "state_trait_glm.csv")
tab.to_csv(tab_path, index=False)
print(f"Saved: {tab_path}")

# Highlight constriction findings
print("")
print("--- Constriction effects ---")
constrict_rows = tab[tab["feature"].isin(["ttr", "root_ttr"])]
for _, row in constrict_rows.iterrows():
    sig = "*" if row["p"] < .05 else "~" if row["p"] < .10 else ""
    print(f"  {row['feature']}_{row['component']}: b={row['beta']:.3f} z={row['z']:.3f} p={row['p']:.3f} {sig}")


=== STATE vs TRAIT (cluster-robust GLM) ===
Outcome: high_any_item_eq3 | N=2385 | participants=123
                 Generalized Linear Model Regression Results                  
Dep. Variable:                      y   No. Observations:                 2385
Model:                            GLM   Df Residuals:                     2365
Model Family:                Binomial   Df Model:                           19
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -188.12
Date:                Fri, 13 Feb 2026   Deviance:                       376.24
Time:                        17:04:09   Pearson chi2:                 3.84e+03
No. Iterations:                     9   Pseudo R-squ. (CS):            0.06858
Covariance Type:              cluster                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------

In [29]:
# =========================
# STATE vs TRAIT GLM — TRIMMED (cluster-robust)
# =========================
# Trimmed predictor set: removed wc_le1 (redundant with log1p_wc); multicollinearity
# and ttr (root_ttr preferred — corrects for length dependence)
N_PCS_CONTENT = 5
OUTCOME_BIN = "high_any_item_eq3"
MECH_COLS = (
    ["log1p_wc", "instability_cosdist", "root_ttr"] +
    [f"PC{i}" for i in range(1, N_PCS_CONTENT + 1)]
)

X_cols = []
for c in MECH_COLS:
    X_cols += [f"{c}_within", f"{c}_between"]

dfm = pm_day.copy()
y_glm = dfm[OUTCOME_BIN].astype(float).values
mask_glm = np.isfinite(y_glm)
for c in list(X_cols):
    if c in dfm.columns:
        mask_glm &= np.isfinite(dfm[c].values)
    else:
        print(f"Warning: {c} not in data, skipping from GLM")
        X_cols = [x for x in X_cols if x != c]

dfm = dfm.loc[mask_glm].copy()
y_glm = dfm[OUTCOME_BIN].astype(int).values
X_glm = sm.add_constant(dfm[X_cols].astype(float), has_constant="add")
groups_glm = dfm[PID_COL].astype(str).values

print(f"Predictors: {X_cols}")
print(f"N = {len(dfm)} | participants = {dfm[PID_COL].nunique()}")
print(f"Positive cases: {y_glm.sum()} ({y_glm.mean():.3f})")
print("")

glm = sm.GLM(y_glm, X_glm, family=sm.families.Binomial())
res = glm.fit(cov_type="cluster", cov_kwds={"groups": groups_glm})

print("=== STATE vs TRAIT GLM (trimmed, cluster-robust) ===")
print(res.summary())

# Save table
rows = []
for c in MECH_COLS:
    for part in ["within", "between"]:
        name = f"{c}_{part}"
        if name in res.params.index:
            rows.append({
                "feature": c, "component": part,
                "beta": float(res.params[name]),
                "SE": float(res.bse[name]),
                "z": float(res.tvalues[name]),
                "p": float(res.pvalues[name]),
                "OR": float(np.exp(res.params[name])),
                "CI_lo": float(res.conf_int().loc[name, 0]),
                "CI_hi": float(res.conf_int().loc[name, 1]),
            })

tab = pd.DataFrame(rows)
tab_path = os.path.join(OUT_DIR, "state_trait_glm.csv")
tab.to_csv(tab_path, index=False)
print(f"Saved: {tab_path}")

# Highlight key findings
print("")
print("--- Significant / trending effects ---")
for _, row in tab.iterrows():
    if row["p"] < .10:
        sig = "**" if row["p"] < .05 else "*" if row["p"] < .10 else ""
        print(f"  {row['feature']}_{row['component']}: b={row['beta']:.3f} z={row['z']:.3f} p={row['p']:.3f} OR={row['OR']:.3f} {sig}")

Predictors: ['log1p_wc_within', 'log1p_wc_between', 'instability_cosdist_within', 'instability_cosdist_between', 'root_ttr_within', 'root_ttr_between', 'PC1_within', 'PC1_between', 'PC2_within', 'PC2_between', 'PC3_within', 'PC3_between', 'PC4_within', 'PC4_between', 'PC5_within', 'PC5_between']
N = 2385 | participants = 123
Positive cases: 58 (0.024)

=== STATE vs TRAIT GLM (trimmed, cluster-robust) ===
                 Generalized Linear Model Regression Results                  
Dep. Variable:                      y   No. Observations:                 2385
Model:                            GLM   Df Residuals:                     2368
Model Family:                Binomial   Df Model:                           16
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -192.15
Date:                Fri, 13 Feb 2026   Deviance:                       384.29
Time:                        17:17:56  